In [1]:
import hydra
from omegaconf import OmegaConf
print('hydra imported')
import os
import torch
from tqdm.auto import tqdm
from datasets.pfams import SyntheticPfamDataset
print('dataset class imported')
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

hydra imported
dataset class imported


In [2]:
output_dir = '/orcd/data/omarabu/001/gokul/CoupledDistributionEmbeddings/outputs/pfam_synthetic_3d28cf5ef445092cb992b8f288d9832c'

config_path = os.path.join(output_dir, 'config.yaml')
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config not found at {config_path}")

config = OmegaConf.load(config_path)

# Detect model types
encoder_type, generator_type = ('esm', 'progen2')

best_model_path = os.path.join(output_dir, 'best_model.pt')
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Best model not foun   d at {best_model_path}")

encoder = hydra.utils.instantiate(config.encoder)
generator = hydra.utils.instantiate(config.generator)

device = 'cuda'
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

encoder.load_state_dict(checkpoint['encoder_state_dict'])
generator.load_state_dict(checkpoint['generator_state_dict'])

epoch = checkpoint.get('epoch', 'unknown')
loss = checkpoint.get('loss', float('nan'))

encoder.to(device)
generator.to(device)
encoder.eval()
generator.eval();

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
pfam_dataset_eval = SyntheticPfamDataset(data_dir='/orcd/data/omarabu/001/gokul/DistributionEmbeddings/data/pfam', 
                           data_file='eval_synthetic_pfam_tokenized_data.pt',
                           tokenize=True,
                           set_size=8,
                           max_length=128)

pfam_dataset_train = SyntheticPfamDataset(data_dir='/orcd/data/omarabu/001/gokul/DistributionEmbeddings/data/pfam', 
                           data_file='synthetic_pfam_tokenized_data.pt',
                           tokenize=True,
                           set_size=8,
                           max_length=128)

In [4]:
# check generator on a train dataset
print('trained for ', epoch, 'epochs')

pair = pfam_dataset_train[100]
print('source canonical kmer:', pair['source_samples']['canonical_kmer'])
print('target canonical kmer:', pair['target_samples']['canonical_kmer'])

source_samples = pair['source_samples']
target_samples = pair['target_samples']

# Move tensors to device and add batch dimension
for key in source_samples.keys():
    if isinstance(source_samples[key], torch.Tensor):
        source_samples[key] = source_samples[key].unsqueeze(0).to(device)
        print(key, source_samples[key].shape)
for key in target_samples.keys():
    if isinstance(target_samples[key], torch.Tensor):
        target_samples[key] = target_samples[key].unsqueeze(0).to(device)

with torch.no_grad():
    source_embedding = encoder(source_samples)
    target_embedding = encoder(target_samples)

    source_samples_for_gen = {
        'progen_input_ids': source_samples['progen_input_ids'][:, 0, :],  # [1, seq_len]
        'progen_attention_mask': source_samples['progen_attention_mask'][:, 0, :],  # [1, seq_len]
    }
    
    generator.temperature = 0.1
    samples = generator.sample(source_samples_for_gen, source_embedding,
                               target_embedding, num_samples=1)

trained for  64 epochs
source canonical kmer: RKHF
target canonical kmer: GTYV
esm_input_ids torch.Size([1, 8, 128])
esm_attention_mask torch.Size([1, 8, 128])
progen_input_ids torch.Size([1, 8, 128])
progen_attention_mask torch.Size([1, 8, 128])


In [5]:
from transformers import AutoTokenizer
progen_tokenizer = AutoTokenizer.from_pretrained('hugohrban/progen2-medium', trust_remote_code=True)
progen_tokenizer.pad_token = '<|pad|>'
progen_tokenizer.bos_token = '<|bos|>'
progen_tokenizer.eos_token = '<|eos|>'

# Decode the samples
# samples shape is [num_samples, seq_len]
decoded_sequences = []
for i in range(samples.shape[1]):
    # Decode each sequence, skipping special tokens
    seq = progen_tokenizer.decode(samples[0, i], skip_special_tokens=True)
    decoded_sequences.append(seq)
    print(f"Sample {i+1}: {seq}")

Sample 1: FKFF


In [6]:
# # concat train and eval embeddings, compute PC and plot
# train_source_embeddings = torch.cat(train_source_embeddings)
# eval_source_embeddings = torch.cat(eval_source_embeddings)
# all_source_embeddings = torch.cat([train_source_embeddings, eval_source_embeddings], dim=0)


# pca = PCA(n_components=2)
# all_source_embeddings_2d = pca.fit_transform(all_source_embeddings)
# train_embeddings_2d = all_source_embeddings_2d[:len(train_source_embeddings)]
# eval_embeddings_2d = all_source_embeddings_2d[len(train_source_embeddings):]

# plt.figure(figsize=(3,3))
# plt.scatter(train_embeddings_2d[:,0], train_embeddings_2d[:,1], label='Train', alpha=0.5)
# plt.scatter(eval_embeddings_2d[:,0], eval_embeddings_2d[:,1], label='Eval', alpha=0.5)
# plt.legend()

In [7]:
from torch.utils.data import default_collate

eval_losses = []
for p in tqdm(range(200)):
    pair = pfam_dataset_eval[p]
    # Wrap the pair in a list to simulate a batch of size 1
    batch = default_collate([pair])
    
    source_samples = batch['source_samples']
    target_samples = batch['target_samples']
    
    # Move to device
    for key in source_samples.keys():
        if isinstance(source_samples[key], torch.Tensor):
            source_samples[key] = source_samples[key].to(device)
    for key in target_samples.keys():
        if isinstance(target_samples[key], torch.Tensor):
            target_samples[key] = target_samples[key].to(device)
    
    with torch.no_grad():
        source_embedding = encoder(source_samples)
        target_embedding = encoder(target_samples)
        output = generator.loss(source_samples, target_samples,
                                source_embedding, target_embedding)
        eval_losses.append(output.item())

  0%|          | 0/200 [00:00<?, ?it/s]

In [8]:
# Check the raw data structure
pair = pfam_dataset_eval[0]
print("Raw pair structure:")
print(f"source progen_input_ids shape: {pair['source_samples']['progen_input_ids'].shape}")
print(f"target progen_input_ids shape: {pair['target_samples']['progen_input_ids'].shape}")

# Check after default_collate
batch = default_collate([pair])
print("\nAfter default_collate:")
print(f"source progen_input_ids shape: {batch['source_samples']['progen_input_ids'].shape}")
print(f"target progen_input_ids shape: {batch['target_samples']['progen_input_ids'].shape}")

# Also check the actual sequence lengths (non-padding)
src_mask = batch['source_samples']['progen_attention_mask']
tgt_mask = batch['target_samples']['progen_attention_mask']
print(f"\nSource actual length (max): {src_mask.sum(dim=-1).max().item()}")
print(f"Target actual length (max): {tgt_mask.sum(dim=-1).max().item()}")
print(f"Combined would be: {src_mask.sum(dim=-1).max().item() + tgt_mask.sum(dim=-1).max().item()}")

# Load real pfam dataset
from datasets.pfams import PfamDataset
pfam_real = PfamDataset(data_dir='/orcd/data/omarabu/001/gokul/DistributionEmbeddings/data/pfam', 
                        data_file='pfam_tokenized_data_small.pt',
                        tokenize=False,
                        set_size=8)

pair_real = pfam_real[0]
print("\nReal Pfam dataset:")
print(f"source progen_input_ids shape: {pair_real['source_samples']['progen_input_ids'].shape}")
print(f"target progen_input_ids shape: {pair_real['target_samples']['progen_input_ids'].shape}")

batch_real = default_collate([pair_real])
print(f"\nAfter collate - source shape: {batch_real['source_samples']['progen_input_ids'].shape}")
print(f"After collate - target shape: {batch_real['target_samples']['progen_input_ids'].shape}")

Raw pair structure:
source progen_input_ids shape: torch.Size([8, 128])
target progen_input_ids shape: torch.Size([8, 128])

After default_collate:
source progen_input_ids shape: torch.Size([1, 8, 128])
target progen_input_ids shape: torch.Size([1, 8, 128])

Source actual length (max): 6
Target actual length (max): 6
Combined would be: 12

Real Pfam dataset:
source progen_input_ids shape: torch.Size([8, 128])
target progen_input_ids shape: torch.Size([8, 128])

After collate - source shape: torch.Size([1, 8, 128])
After collate - target shape: torch.Size([1, 8, 128])


In [9]:
from sklearn.neighbors import NearestNeighbors

# fit top 1 nearest neighbor on train embeddings
nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(train_source_embeddings[:1])


NameError: name 'train_source_embeddings' is not defined

In [ ]:
snap_to_nn_losses = []

for p in tqdm(range(400)):
    pair = pfam_dataset_eval[p]
    source_samples = pair['source_samples']
    print(source_samples['canonical_kmer'])
    target_samples = pair['target_samples']
    for key in source_samples.keys():
        if type(source_samples[key]) == torch.Tensor:
            source_samples[key] = source_samples[key].unsqueeze(0).to(device)
    for key in target_samples.keys():
        if type(target_samples[key]) == torch.Tensor:
            target_samples[key] = target_samples[key].unsqueeze(0).to(device)
    with torch.no_grad():
        source_embedding = encoder(source_samples)
        # find nearest neighbor in train set
        distances, indices = nbrs.kneighbors(source_embedding.cpu())
        nn_index = indices[0][0]
        nn_embedding = train_source_embeddings[nn_index].unsqueeze(0).to(device)

        target_embedding = encoder(target_samples)
        # find nearest neighbor in train set
        distances, indices = nbrs.kneighbors(target_embedding.cpu())
        nn_index = indices[0][0]
        nn_embedding = train_source_embeddings[nn_index].unsqueeze(0).to(device)


        output = generator.loss(source_samples, target_samples,
                                nn_embedding, target_embedding)
        
        snap_to_nn_losses.append(output.item())

  0%|          | 0/400 [00:00<?, ?it/s]

In [ ]:
print(np.mean(eval_losses), np.mean(snap_to_nn_losses))

2.8243811345100402 2.824858309030533
